# Chapter 1 Lab: The Imperative of Performance in Modern AI

**Dr. Kaoutar El Maghraoui**

Chapter 1 is mostly argument. Why performance has to be a design concern from day one, what the four factors are (latency, throughput, cost, reliability), and how the running RAG case study fits together. This notebook is where you stop reading about those ideas and start measuring them on hardware you actually have.

By the end you will have:

1. An auto-detected **hardware tier**, matching one of the four capability classes in Figure 1.10 (consumer CPU, consumer GPU 16 GB, datacenter GPU 40 GB, datacenter GPU 80 GB+).
2. A live **latency-throughput trade-off curve** (Figure 1.4) measured on your own GPU with a small matmul workload, the three operating zones shaded in: real-time, interactive, batch.
3. A working **naïve RAG baseline** matching the architecture in the *System architecture overview* section: embedding plus FAISS plus a small instruct LM. You will reuse this exact baseline and improve it through Chapters 2 to 9.
4. An end-to-end **latency waterfall** for a single query, reproducing the structure of Table 1.2 (query encode → retrieval → context → prefill → decode).
5. A short **accuracy-performance gap** demo. Same model, same prompt, same answer, very different cost per query.
6. Answers to the **Knowledge Check** at the end of the chapter.

### Before you start

Set **Runtime → Change runtime type → Hardware accelerator: T4 GPU.** The notebook will run on CPU too, but each LLM decode takes 30 to 60 seconds instead of 3 to 8, and the waterfall ends up dominated by CPU-bound matmul rather than the bottleneck the chapter is actually about.

Total runtime: about 12 minutes on a Colab Free T4, about 25 minutes on a CPU laptop.

## 0  Setup

Install the stack. The `-q` flag keeps the output tidy. About 90 seconds on a cold Colab runtime.

In [ ]:
# Tested stack for the book (Chapters 1-5 and 7): Python 3.11, PyTorch 2.14.1 (CUDA 13.0),
# transformers 5.20.2, sentence-transformers 6.2.0. Colab ships its own PyTorch build with CUDA,
# so torch is not reinstalled here (installing an older wheel would downgrade the runtime).
# The library versions match the Chapter 2 lab, so nothing has to be reinstalled later.
!pip install -q transformers==5.20.2 sentence-transformers==6.2.0 accelerate faiss-cpu datasets bitsandbytes

## 1  Hardware tier auto-detection (Figure 1.10)

The case study targets four capability classes, not specific chips. Same pipeline across all of them. Only the model sizes change. The cell below picks the right tier for what you are running on, so the rest of the notebook scales itself.

In [ ]:
import torch, subprocess, sys, platform

def detect_tier():
    """Return one of: cpu, gpu_16gb, gpu_40gb, gpu_80gb (Figure 1.10 capability classes)."""
    if not torch.cuda.is_available():
        return 'cpu'
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb < 24:    return 'gpu_16gb'   # T4 (15 GB), L4 (24 GB rounded down), RTX 4060 Ti 16 GB
    if vram_gb < 60:    return 'gpu_40gb'   # A100 40 GB, L40 (48 GB)
    return 'gpu_80gb'                      # A100 80 GB, H100 80 GB, H200 141 GB

TIER = detect_tier()

# Figure 1.10 model recipe per tier. Model names are 2026 exemplars;
# the structural property (same pipeline, scaled model size) is what
# matters and persists as exemplars rotate.
RECIPE = {
    'cpu':      {'embed': 'sentence-transformers/all-MiniLM-L6-v2', 'embed_dim': 384,  'lm': 'Qwen/Qwen2.5-0.5B-Instruct',     'lm_4bit': False, 'expected_ttft_s': '10-30'},
    'gpu_16gb': {'embed': 'sentence-transformers/all-MiniLM-L6-v2', 'embed_dim': 384,  'lm': 'Qwen/Qwen2.5-1.5B-Instruct',     'lm_4bit': True,  'expected_ttft_s': '3-8'},
    'gpu_40gb': {'embed': 'BAAI/bge-base-en-v1.5',                  'embed_dim': 768,  'lm': 'Qwen/Qwen2.5-3B-Instruct',       'lm_4bit': False, 'expected_ttft_s': '1-3'},
    'gpu_80gb': {'embed': 'BAAI/bge-large-en-v1.5',                 'embed_dim': 1024, 'lm': 'Qwen/Qwen2.5-7B-Instruct',       'lm_4bit': False, 'expected_ttft_s': '0.5-1'},
}[TIER]

print(f'Detected tier: {TIER}')
print(f'Python:        {sys.version.split()[0]}')
print(f'PyTorch:       {torch.__version__}')
print(f'Platform:      {platform.platform()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU:           {torch.cuda.get_device_name(0)}  ({p.total_memory/1e9:.1f} GB HBM, {p.multi_processor_count} SMs, sm_{p.major}{p.minor})')
else:
    print('GPU:           none. Falling back to CPU recipe')
print()
print(f'Embedding model:    {RECIPE["embed"]}  ({RECIPE["embed_dim"]}-d)')
print(f'Generation model:   {RECIPE["lm"]}{"  (4-bit)" if RECIPE["lm_4bit"] else ""}')
print(f'Expected naïve TTFT for tier {TIER}: ~{RECIPE["expected_ttft_s"]} s')

If you got `gpu_16gb` and you are on Colab Free, you are on the default tier the rest of the book targets. Every absolute number quoted in Table 1.2 is for this tier. If you got something else, your absolute numbers will rescale, but the **speedup ratios** the chapters teach you to recover (4.6× by the end of Chapter 9) are roughly stable across tiers. That stability is the durable claim of the *System architecture overview* section.

## 2  The four factors made concrete: a latency-throughput curve (Figure 1.4)

Figure 1.4 in the chapter shows latency rising as batch size grows while throughput rises with it. The same trade-off curve governs every serving system you will deploy. We will reproduce it on your hardware with a small matmul that stands in for an embedding model's forward pass.

Wall-clock latency *per request* and aggregate throughput, swept from batch size 1 to 64. The shape of the curve is what matters here. Absolute numbers depend on your tier and are not the point.

In [ ]:
import time, torch
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# A request = one forward pass through a fake 768→3072→768 MLP.
# Roughly the per-token cost of a single transformer block at small scale.
torch.manual_seed(0)
W1 = torch.randn(768, 3072, device=device, dtype=torch.float16 if device=='cuda' else torch.float32)
W2 = torch.randn(3072, 768, device=device, dtype=torch.float16 if device=='cuda' else torch.float32)

def step(batch_size):
    x = torch.randn(batch_size, 768, device=device, dtype=W1.dtype)
    return torch.relu(x @ W1) @ W2

def benchmark(batch_size, n_iter=20, warmup=3):
    for _ in range(warmup):
        y = step(batch_size)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_iter):
        y = step(batch_size)
    if device == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    batch_latency_ms = (elapsed / n_iter) * 1000
    per_request_latency_ms = batch_latency_ms                    # in a batched system every request waits for the full batch
    throughput_rps = batch_size / (batch_latency_ms / 1000)
    return per_request_latency_ms, throughput_rps

batch_sizes = [1, 2, 4, 8, 16, 32, 64]
results = [(bs, *benchmark(bs)) for bs in batch_sizes]

print(f'{"BS":>4}  {"per-req latency (ms)":>22}  {"throughput (req/s)":>20}')
for bs, lat, tput in results:
    print(f'{bs:>4}  {lat:>22.2f}  {tput:>20.1f}')

# --- self-check: throughput rises with batch size and timings are sane ---
_tputs = [tp for _, _, tp in results]
_lats  = [la for _, la, _ in results]
assert all(v > 0 for v in _tputs + _lats), 'benchmark produced a non-positive timing'
assert _tputs[-1] > _tputs[0], f'throughput should rise with batch size, got {_tputs}'
print(f'self-check passed: throughput {_tputs[0]:.0f} -> {_tputs[-1]:.0f} req/s across batch sizes')


In [ ]:
# Plot the curve. Three operating zones from Figure 1.4 are shaded.
import matplotlib.pyplot as plt

lat = [r[1] for r in results]
tput = [r[2] for r in results]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(tput, lat, 'o-', linewidth=2, color='#B85C2C', markersize=8, zorder=5)
for (bs, lat_v, tput_v) in results:
    ax.annotate(f'BS={bs}', (tput_v, lat_v), textcoords='offset points', xytext=(8, 8), fontsize=9)

# Operating zones. Boundaries match Figure 1.4 in the chapter
ymax = max(max(lat) * 1.15, 1.0)
# Zone boundaries from Figure 1.4, clamped to the visible range so a fast GPU
# with sub-millisecond latencies does not push labels off the axis.
zones = [
    (0,   75,   'green', 0.10, 'Real-time (fraud, search)',    '#1f6b1f'),
    (75,  150,  'gold',  0.10, 'Interactive (chatbots)',       '#7a5a00'),
    (150, ymax, 'red',   0.08, 'Batch (indexing, offline)',    '#7a1f1f'),
]
for lo, hi, color, alpha, label, tcolor in zones:
    if lo >= ymax:
        continue
    hi = min(hi, ymax)
    ax.axhspan(lo, hi, alpha=alpha, color=color, zorder=0)
    ax.text(max(tput)*0.98, (lo + hi) / 2, label, ha='right', va='center', fontsize=9, color=tcolor)

ax.set_xlabel('Throughput (requests / second)')
ax.set_ylabel('Per-request latency (ms)')
ax.set_title('Latency-throughput trade-off on your hardware  (reproduces Figure 1.4)')
ax.set_ylim(0, ymax)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Read the curve. Going from BS=1 to BS=64 typically multiplies throughput by 5 to 15× while only multiplying per-request latency by 2 to 3×. That asymmetry is why batching exists at all. Throughput grows faster than latency does.

But the curve also bends. Past some batch size, latency starts rising faster than throughput does, and you slide out of the interactive zone into the batch zone. This is the **Pareto frontier** the *Throughput* section talks about, and Chapter 8 is largely about how to pick where to sit on it.

## 3  Build the naïve RAG baseline (*System architecture overview*)

Now the case-study system itself. **This is the unoptimized baseline** that motivates every chapter that follows. Three pieces, matching Figure 1.10:

1. **Indexing (offline):** chunk a small corpus, embed each chunk, store in FAISS.
2. **Retrieval (online):** encode the query with the same embedding model, ANN-search the top-k.
3. **Generation (online):** prompt the LM with the retrieved chunks, generate an answer.

A 200-paragraph Wikipedia subset is enough for the pipeline to run end-to-end, and indexing finishes in under a minute on any tier.

In [ ]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import faiss, numpy as np

# Tiny corpus. 200 short Wikipedia paragraphs is enough to demonstrate the pipeline.
ds = load_dataset('rag-datasets/rag-mini-wikipedia', 'text-corpus', split='passages')
ds = ds.select(range(200))
corpus = [row['passage'] for row in ds]
print(f'Corpus loaded: {len(corpus)} passages, mean length {int(np.mean([len(p.split()) for p in corpus]))} tokens')

embedder = SentenceTransformer(RECIPE['embed'], device=device)
embeddings = embedder.encode(corpus, batch_size=32, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

index = faiss.IndexFlatIP(RECIPE['embed_dim'])   # exact inner-product search; we'll see in Ch.6 why this scales poorly
index.add(embeddings.astype(np.float32))
print(f'FAISS index: {index.ntotal} vectors, dim={RECIPE["embed_dim"]}')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

lm_kwargs = {'torch_dtype': torch.float16, 'device_map': 'auto'} if device == 'cuda' else {'torch_dtype': torch.float32}
if RECIPE['lm_4bit'] and device == 'cuda':
    from transformers import BitsAndBytesConfig
    lm_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tok = AutoTokenizer.from_pretrained(RECIPE['lm'])
lm  = AutoModelForCausalLM.from_pretrained(RECIPE['lm'], **lm_kwargs)
if device == 'cpu': lm = lm.to('cpu')
lm.eval()
print(f'Loaded {RECIPE["lm"]}')

In [ ]:
@torch.inference_mode()
def naive_rag(query, top_k=3, max_new_tokens=64):
    """Naïve baseline. Every stage is unoptimized on purpose. That is the point of Chapter 1."""
    q_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    _, idx = index.search(q_emb, top_k)
    retrieved = [corpus[i] for i in idx[0]]
    context = '\n\n'.join(f'[{i+1}] {p}' for i, p in enumerate(retrieved))
    prompt = f'Use the context to answer.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:'
    inputs = tok(prompt, return_tensors='pt').to(lm.device)
    out = lm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    answer = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return answer

answer = naive_rag('Who wrote Hamlet?')
print('Q: Who wrote Hamlet?')
print(f'A: {answer.strip()}')

## 4  The latency waterfall (Table 1.2)

Time each stage of the pipeline separately. This is the **stage breakdown** Table 1.2 reports for a T4 GPU. If you are on the `gpu_16gb` tier, your numbers should be in the same neighborhood. On other tiers they rescale, but the *shape* of the breakdown holds.

In [ ]:
import time

@torch.inference_mode()
def naive_rag_timed(query, top_k=3, max_new_tokens=20):
    timings = {}
    if device == 'cuda': torch.cuda.synchronize()

    # Stage 1: query encoding
    t = time.perf_counter()
    q_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    if device == 'cuda': torch.cuda.synchronize()
    timings['query_encode_ms'] = (time.perf_counter() - t) * 1000

    # Stage 2: ANN retrieval
    t = time.perf_counter()
    _, idx = index.search(q_emb, top_k)
    timings['retrieval_ms'] = (time.perf_counter() - t) * 1000

    # Stage 3: context assembly
    t = time.perf_counter()
    retrieved = [corpus[i] for i in idx[0]]
    context = '\n\n'.join(f'[{i+1}] {p}' for i, p in enumerate(retrieved))
    prompt = f'Use the context to answer.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:'
    timings['context_assembly_ms'] = (time.perf_counter() - t) * 1000

    # Stage 4: LLM prefill (just the prompt forward pass)
    inputs = tok(prompt, return_tensors='pt').to(lm.device)
    if device == 'cuda': torch.cuda.synchronize()
    t = time.perf_counter()
    out_prefill = lm(**inputs)
    if device == 'cuda': torch.cuda.synchronize()
    timings['llm_prefill_ms'] = (time.perf_counter() - t) * 1000

    # Stage 5: LLM decode (generate N new tokens)
    t = time.perf_counter()
    out = lm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    if device == 'cuda': torch.cuda.synchronize()
    timings['llm_decode_ms'] = (time.perf_counter() - t) * 1000

    timings['total_ms'] = sum(v for k, v in timings.items() if k.endswith('_ms'))
    timings['answer'] = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return timings

# Warmup. First call always carries CUDA-init / autotune costs that don't represent steady state.
_ = naive_rag_timed('What is the capital of France?')

# Real measurement
t = naive_rag_timed('What is the capital of France?')
print(f'Q: What is the capital of France?')
print(f'A: {t["answer"]}')
print()
print(f'{"Stage":<24}{"Time (ms)":>12}    Bottleneck class       Primary technique (chapter)')
print('-' * 100)
rows = [
    ('Query encoding',       t['query_encode_ms'],     'Compute-bound',           'torch.compile, quantization (Ch. 3-4, 9)'),
    ('ANN retrieval',        t['retrieval_ms'],        'Memory-bound',            'Index optimization, GPU FAISS (Ch. 6)'),
    ('Context assembly',     t['context_assembly_ms'], 'I/O-bound',               'Caching, async prefetch (Ch. 8)'),
    ('LLM prefill',          t['llm_prefill_ms'],      'Compute-bound',           'FlashAttention, torch.compile (Ch. 4-5, 8)'),
    ('LLM decode (20 toks)', t['llm_decode_ms'],       'Memory-bandwidth bound',  'Speculative decoding, KV quant. (Ch. 8-9)'),
]
for stage, ms, btype, tech in rows:
    print(f'{stage:<24}{ms:>12.1f}    {btype:<22} {tech}')
print('-' * 100)
print(f'{"Total":<24}{t["total_ms"]:>12.1f}    LLM decode dominates'.ljust(80))
print(f'\nDecode share of total: {t["llm_decode_ms"] / t["total_ms"] * 100:.1f}%')

# --- self-check: decode dominates the naive pipeline ---
_stage_ms = {k: v for k, v in t.items() if k.endswith('_ms') and k != 'total_ms'}
assert t['total_ms'] > 0, 'total latency should be positive'
assert max(_stage_ms, key=_stage_ms.get) == 'llm_decode_ms', f'expected decode to dominate, got {_stage_ms}'
assert t['llm_decode_ms'] / t['total_ms'] > 0.3, f"decode share unexpectedly low: {t['llm_decode_ms']/t['total_ms']*100:.0f}%"
print(f"self-check passed: decode is the largest stage ({t['llm_decode_ms']/t['total_ms']*100:.0f}% of total)")


In [ ]:
# Visualize as a stacked bar. The chapter calls this a 'latency waterfall'.
import matplotlib.pyplot as plt

stages = ['Query\nencoding', 'ANN\nretrieval', 'Context\nassembly', 'LLM\nprefill', 'LLM\ndecode']
values = [t['query_encode_ms'], t['retrieval_ms'], t['context_assembly_ms'], t['llm_prefill_ms'], t['llm_decode_ms']]
colors = ['#9CC9E0', '#7FB3D5', '#F5B041', '#E67E22', '#B85C2C']

fig, ax = plt.subplots(figsize=(10, 2.5))
left = 0
for s, v, c in zip(stages, values, colors):
    ax.barh([0], [v], left=left, color=c, edgecolor='white', linewidth=1.5)
    if v > t['total_ms'] * 0.04:
        ax.text(left + v/2, 0, f'{s}\n{v:.0f} ms', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    left += v
ax.set_xlim(0, t['total_ms'] * 1.02)
ax.set_xlabel('Time (ms)')
ax.set_yticks([])
ax.set_title(f'Naïve RAG latency waterfall, total {t["total_ms"]:.0f} ms on tier `{TIER}`  (reproduces structure of Table 1.2)')
plt.tight_layout()
plt.show()

Two things to notice.

**The bar is dominated by LLM decode.** On the `gpu_16gb` tier, decode usually accounts for more than 60% of total wall time. The chapter argues at length that this is *the* reason inference dominates lifecycle cost (*Cost*) and *the* reason performance engineering matters in the first place. If you had skipped the measurement and gone straight to optimizing the vector retrieval (a tempting first move), the user would have seen no change at all.

**Decode is memory-bandwidth bound, not compute-bound.** *The GenAI inference problem* section explains why. Each generated token forces the full model weight matrix to be loaded from HBM, and the math per token is small relative to the bytes moved. That structural property is what makes speculative decoding (Ch. 8) and weight quantization (Ch. 9) the highest-payoff techniques in the book.

Hold this baseline number in your head. By the end of Chapter 9 you will have driven it down by about 4.6× on the same hardware.

## 5  The accuracy-performance gap, in one experiment (*The accuracy–performance gap*)

The chapter argues, with the Booking.com and Microsoft Bing/BERT case studies, that a model can be entirely correct and entirely useless at the same time. Production economics, not benchmark accuracy, decide whether it ships.

A one-cell version of that argument: run the same RAG system as a serial stream and project the cost per query at scale. The model has not changed. The answers have not changed. But the dollar figures absolutely have.

In [ ]:
import time

queries = [
    'Who wrote Hamlet?',
    'What is the capital of France?',
    'Who painted the Mona Lisa?',
    'What year did World War II end?',
    'Who discovered penicillin?',
    'What is the largest planet in the solar system?',
    'Who composed the Ninth Symphony?',
    'What is the boiling point of water in Celsius?',
]

# Single-request mode (interactive)
if device == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter()
for q in queries:
    _ = naive_rag(q, max_new_tokens=20)
if device == 'cuda': torch.cuda.synchronize()
t_serial = time.perf_counter() - t0

# Naïvely 'batched'. We just measure throughput; real batching belongs to Ch. 8.
# This is a thought experiment about cost economics, not a prescription.
per_query_serial_s = t_serial / len(queries)
throughput_serial  = len(queries) / t_serial

# Use a hypothetical T4-on-Colab cost as a stand-in for the cloud GPU rate.
# The 'cost per useful output' calculation from Chapter 1's *Cost* section.
GPU_PRICE_PER_HOUR = 0.40           # T4 spot ≈ $0.35-0.45/hr in late 2025
cost_per_query_usd = (per_query_serial_s / 3600) * GPU_PRICE_PER_HOUR
print(f'Serial run of {len(queries)} queries:')
print(f'  total wall-clock:       {t_serial:.2f} s')
print(f'  per-query latency:      {per_query_serial_s*1000:.0f} ms')
print(f'  throughput:             {throughput_serial:.2f} queries/s')
print(f'  cost per query:         ${cost_per_query_usd:.5f}  (at ${GPU_PRICE_PER_HOUR}/hr)')
print()
print(f'At 10 M queries/day (Bing-scale):  ${cost_per_query_usd * 10_000_000:,.0f}/day')
print(f'At 100 M queries/day (ChatGPT-scale): ${cost_per_query_usd * 100_000_000:,.0f}/day')

Look at the dollar figures at the bottom. The model has not changed, the prompt has not changed, the answer has not changed. But the cost per query you just measured, projected to ChatGPT scale, is the difference between a viable product and one that loses money on every query. **That is what the *Cost* section's claim that inference is 90% of lifecycle compute cost actually means in practice.** Every percentage point of efficiency is recurring revenue.

And we have not done any of the optimization work yet. Just the techniques in Chapters 3, 4, 8, and 9 will move this number by an order of magnitude.

## 6  Knowledge check

Six questions from the end-of-chapter Knowledge Check. Try answering them in your own words *before* expanding the suggested answers.

---

**Q1.** A fraud detection model achieves 98.2% accuracy but requires 500 ms per inference. The system SLA requires sub-50 ms response time. (a) How would you frame this as a performance engineering problem? (b) Based on the performance engineering concepts introduced in this chapter, what are the first two diagnostic questions you would ask before attempting any optimization, and why?

<details><summary>Suggested answer</summary>

(a) The model has cleared the **accuracy bar** but missed the **latency SLA**. Per the *Translating lab accuracy to production value* section, accuracy is necessary but not sufficient. The job is to recover the missing 10× in latency without losing measurable accuracy. Closing the accuracy-performance gap (*The accuracy–performance gap*).

(b) **Performance Engineering Principle #1 (Measure):** *where is the time actually going?* Without an end-to-end profile (Chapter 2), you do not know whether the bottleneck is feature engineering, inference, or post-processing. **Principle #2 (Define targets):** *what specific latency budget per stage?* If 500 ms is the budget for the model, but the SLA is 50 ms end-to-end including network and feature lookup, the real model budget might be 30 ms. Optimizing without those numbers is guessing.

</details>

---

**Q2.** Explain the difference between latency and throughput in the context of an LLM serving system. Describe a specific scenario where optimizing for throughput necessarily increases latency, and explain the mechanism by which it does so.

<details><summary>Suggested answer</summary>

**Latency** is per-request wall time. **Throughput** is aggregate work per unit time (RPS or tokens/s). They are not inverses (*Throughput*). Continuous batching is the canonical scenario where throughput rises and per-request latency rises together. Each new request has to wait for the next batch boundary before it can start, and once batched, its forward pass shares GPU time with its batch peers. Padding to the longest sequence in the batch wastes additional compute. The mechanism is queueing delay plus shared compute, not algorithmic inefficiency. You saw this empirically in Section 2 of this notebook.

</details>

---

**Q3.** For a production chatbot serving 10,000 concurrent users, rank the four performance factors (latency, throughput, cost, reliability) in order of priority and justify your ranking. Now consider a nightly batch pipeline that summarizes 1 million documents overnight. How does your ranking change, and why?

<details><summary>Suggested answer</summary>

**Chatbot:** Latency ≥ Reliability > Throughput > Cost. TTFT under 2 s is non-negotiable (*Latency*, Booking.com). Reliability ties for first because output incoherence at high concurrency is user-facing. Throughput drives capacity but is not user-facing. Cost is bounded but rarely the primary lever.

**Nightly batch:** Throughput ≥ Cost > Reliability > Latency. Per-request latency does not matter. Only end-of-night completion matters. Throughput and cost dominate because the workload *is* the GPU bill. Reliability matters but checkpointing lets you recover. Latency drops to lowest priority. The inversion of the chatbot.

Table 1.1 lays out the same logic across six application classes.

</details>

---

**Q4.** Why is the LLM decode stage memory-bandwidth bound rather than compute-bound, and what does that imply about the bottleneck? (Refer to *The GenAI inference problem* and your measurements in Section 4 of this notebook.)

<details><summary>Suggested answer</summary>

Each decode step processes one token (sometimes a small handful with speculative decoding). To compute that token, the GPU has to load the *entire* model weight matrix from HBM into the SMs, plus the growing KV cache. The arithmetic per token is small relative to the bytes moved, so the compute units idle while waiting on memory. That is the textbook definition of a memory-bandwidth-bound workload. See Chapter 5's Roofline model for the formal version. The implication: throwing more FLOPs at it (a faster GPU with the same HBM bandwidth) buys very little. The payoff is in moving fewer bytes per token. Weight quantization (4-bit or 8-bit), KV cache compression, speculative decoding (more tokens per HBM round trip).

</details>

---

**Q5.** Training GPT-4 cost approximately $100 M. If inference is ~90% of total lifecycle compute cost, and the model serves 10 M queries/day at $0.01/query, how long until inference cost exceeds training cost? What does this imply about engineering investment?

<details><summary>Suggested answer</summary>

Daily inference cost = 10 M × $0.01 = $100 K. Days to reach training cost = $100 M / $100 K = 1,000 days, about 2.7 years. But a 10% inference efficiency improvement saves $10 K/day, $3.65 M/year, every year forever. A 50% efficiency improvement (achievable with the techniques in Chapters 8 and 9, quantization plus continuous batching) saves $18 M/year. **A one-time training-cost reduction pays once. An inference-efficiency improvement pays every day the model is live.** That is why the book spends most of its pages on inference, not training (*Cost*, Figure 1.7).

</details>

---

**Q6.** Identify two specific differences between traditional HPC (e.g. finite-element simulation) and HPML (e.g. attention computation) that change how you optimize a GPU kernel. Consider memory access patterns, data types, and parallelism structure.

<details><summary>Suggested answer</summary>

**Numerical precision (*HPML versus traditional HPC*).** HPC needs FP64 for solver stability. Losing it changes the answer. ML training tolerates BF16/FP16, and inference often INT8 or INT4, because gradient noise from minibatching dominates the precision floor. Modern accelerators deliver about 16× more throughput at BF16 than at FP64, which is why HPML kernels are designed around the lower-precision tensor cores while HPC kernels still target the FP64 pipes.

**Memory access patterns.** HPC stencils touch a regular grid with predictable neighborhoods. Perfect for tiling and prefetch. Attention computation is data-dependent. Which tokens attend to which is decided by softmax-weighted dot products at runtime, and the KV cache grows token by token. That irregularity defeats classical loop-tiling, and is the reason FlashAttention had to be invented as a fundamentally different fused-kernel pattern (Chapters 5 and 6).

**Bonus, parallelism taxonomy.** HPC almost exclusively uses domain decomposition (data parallelism). HPML adds tensor parallelism (within-layer) and pipeline parallelism (across-layer) to shard model weights that do not fit on one device. A richer parallelism algebra (Chapter 7).

</details>

## What you have just done

You started this notebook with four conceptual factors (latency, throughput, cost, reliability) and ended it with a measurable, working RAG system on your own hardware whose dominant bottleneck you can name and whose dollar cost you can project.

Chapter 2 turns that informal stage timing into a real methodology: PyTorch profiler, Nsight Systems, the Roofline Model, regression tests, and a written performance specification. Chapters 3 and 4 start collapsing the LLM decode stage with `torch.compile` and quantization. By Chapter 9 this 5 to 8 second baseline is under 1.5 seconds end-to-end on the same T4. The 4.6× speedup in Figure 1.11.

Save your end-to-end latency from Section 4. Chapter 2's regression test compares against it.

**Next:** open `chapter02_lab.ipynb`.